# Experiment 2: LLaDA アンマスク可視化

拡散型言語モデル LLaDA の推論過程で、どのトークンがどのステップでアンマスクされるかを記録し、
品詞別に集計・可視化する。Experiment 1 の BERT 実験との比較を行う。

**モデル**: `GSAI-ML/LLaDA-8B-Instruct` (bfloat16, ~16GB)  
**代替**: `inclusionAI/LLaDA-MoE-7B-A1B-Instruct` (メモリ不足時)  
**実行環境**: Google Colab (A100 推奨 / T4 でも動作可能)  
**参考**: https://github.com/ML-GSAI/LLaDA

## 1. 環境セットアップ

In [ ]:
# LLaDA 公式指定の transformers バージョン + 形態素解析・可視化ライブラリ
!pip install -q transformers==4.38.2 accelerate fugashi unidic-lite matplotlib pandas

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import fugashi
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer

# GPU 確認
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    raise RuntimeError("GPU が検出されません。ランタイム → ランタイムのタイプを変更 → GPU を選択してください。")

print("セットアップ完了")

## 2. 設定

In [ ]:
# ===== モデル選択 =====
# A100 (40GB+) → LLaDA-8B を使用
# T4 (16GB) → メモリ不足時は MoE に切り替え

gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
if gpu_mem >= 30:
    MODEL_ID = "GSAI-ML/LLaDA-8B-Instruct"
    print(f"GPU メモリ {gpu_mem:.0f}GB → LLaDA-8B を使用")
else:
    MODEL_ID = "inclusionAI/LLaDA-MoE-7B-A1B-Instruct"
    print(f"GPU メモリ {gpu_mem:.0f}GB → LLaDA-MoE を使用（省メモリ版）")

MASK_ID = 126336  # LLaDA の [MASK] トークンID

# 生成パラメータ
GEN_LENGTH = 64       # 生成トークン数
STEPS = 64            # 拡散ステップ数（= GEN_LENGTH で各ステップ1トークンずつアンマスク）
BLOCK_LENGTH = 64     # = GEN_LENGTH（単一ブロック: 純粋な拡散過程を観察）
TEMPERATURE = 0.      # 決定的生成（再現性のため）
CFG_SCALE = 0.        # Classifier-free guidance なし

# テストプロンプト（Exp1 のテスト文に対応するテーマ）
TEST_PROMPTS = [
    "コーヒーの美味しさについて日本語で一文だけ書いてください。",
    "東京の桜の美しさについて日本語で一文だけ書いてください。",
    "朝の運動習慣について日本語で一文だけ書いてください。",
    "おすすめの本について日本語で一文だけ書いてください。",
    "雨の日の過ごし方について日本語で一文だけ書いてください。",
]

# Experiment 1 の BERT 結果（比較用ベースライン）
# 品詞別の正規化平均アンマスクステップ（0=最初, 1=最後）
# 元データ: 助詞4.5, 動詞6.4, 名詞6.7, 助動詞7.2, 形容詞8.7, 副詞12.0（全60トークン中）
BERT_BASELINE = {
    "助詞":   4.5 / 12.0,  # 正規化
    "動詞":   6.4 / 12.0,
    "名詞":   6.7 / 12.0,
    "助動詞": 7.2 / 12.0,
    "形容詞": 8.7 / 12.0,
    "副詞":   12.0 / 12.0,
}

# 内容語 vs 機能語
CONTENT_POS = {"名詞", "動詞", "形容詞", "副詞"}

print(f"モデル: {MODEL_ID}")
print(f"生成長: {GEN_LENGTH} tokens, ステップ: {STEPS}")

## 3. モデルのロード

In [ ]:
print(f"モデルをロード中: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
).to("cuda").eval()
print(f"✓ ロード完了 (dtype: {model.dtype})")
print(f"  MASK token ID: {MASK_ID}")
print(f"  vocab size: {tokenizer.vocab_size}")

## 4. generate 関数の改造（アンマスクログ付き）

LLaDA 公式の `generate()` をベースに、各拡散ステップで
「どの位置が新たにアンマスクされたか」「そのトークンは何か」「確信度はいくらか」を記録する。

In [ ]:
def add_gumbel_noise(logits, temperature):
    """Gumbel-max サンプリング用のノイズ付与（LLaDA 公式準拠）"""
    if temperature == 0:
        return logits
    logits = logits.to(torch.float64)
    noise = torch.rand_like(logits, dtype=torch.float64)
    gumbel_noise = (- torch.log(noise)) ** temperature
    return logits.exp() / gumbel_noise


def get_num_transfer_tokens(mask_index, steps):
    """各ステップでアンマスクするトークン数を事前計算（線形スケジュール）"""
    mask_num = mask_index.sum(dim=1, keepdim=True)
    base = mask_num // steps
    remainder = mask_num % steps
    num_transfer_tokens = torch.zeros(
        mask_num.size(0), steps, device=mask_index.device, dtype=torch.int64
    ) + base
    for i in range(mask_num.size(0)):
        num_transfer_tokens[i, :remainder[i]] += 1
    return num_transfer_tokens

In [ ]:
@torch.no_grad()
def generate_with_logging(
    model, prompt, steps=64, gen_length=64, block_length=64,
    temperature=0., cfg_scale=0., remasking="low_confidence",
    mask_id=126336,
):
    """
    LLaDA の generate 関数にアンマスクログを追加した版

    Returns:
        x: 生成結果のトークンID列 (1, prompt_len + gen_length)
        unmask_log: list[dict]
            各エントリ: {
                'step': int,           # 拡散ステップ番号 (1-indexed)
                'position': int,       # トークン位置（prompt含む絶対位置）
                'gen_position': int,   # 生成領域内の相対位置 (0-indexed)
                'token_id': int,       # アンマスクされたトークンID
                'confidence': float,   # そのトークンの予測確信度
            }
        prompt_len: int  # プロンプトの長さ
    """
    prompt_len = prompt.shape[1]
    x = torch.full(
        (1, prompt_len + gen_length), mask_id, dtype=torch.long, device=model.device
    )
    x[:, :prompt_len] = prompt.clone()

    prompt_index = (x != mask_id)

    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    steps_per_block = steps // num_blocks

    unmask_log = []
    global_step = 0

    for num_block in range(num_blocks):
        block_start = prompt_len + num_block * block_length
        block_end = prompt_len + (num_block + 1) * block_length
        block_mask_index = (x[:, block_start:block_end] == mask_id)
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps_per_block)

        for i in range(steps_per_block):
            global_step += 1
            mask_index = (x == mask_id)

            # モデル推論
            if cfg_scale > 0.:
                un_x = x.clone()
                un_x[prompt_index] = mask_id
                x_ = torch.cat([x, un_x], dim=0)
                logits = model(x_).logits
                logits, un_logits = torch.chunk(logits, 2, dim=0)
                logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
            else:
                logits = model(x).logits

            logits_with_noise = add_gumbel_noise(logits, temperature=temperature)
            x0 = torch.argmax(logits_with_noise, dim=-1)  # (1, L)

            # 確信度計算
            if remasking == "low_confidence":
                p = F.softmax(logits.float(), dim=-1)
                x0_p = torch.squeeze(
                    torch.gather(p, dim=-1, index=torch.unsqueeze(x0, -1)), -1
                )  # (1, L)
            elif remasking == "random":
                x0_p = torch.rand((x0.shape[0], x0.shape[1]), device=x0.device)
            else:
                raise NotImplementedError(remasking)

            # 現在のブロックより後ろの位置は無視
            x0_p[:, block_end:] = -np.inf

            # マスク位置のみを対象に
            x0 = torch.where(mask_index, x0, x)
            confidence = torch.where(mask_index, x0_p, -np.inf)

            # top-k 選択
            transfer_index = torch.zeros_like(x0, dtype=torch.bool)
            for j in range(confidence.shape[0]):
                k = num_transfer_tokens[j, i].item()
                if k > 0:
                    _, select_index = torch.topk(confidence[j], k=k)
                    transfer_index[j, select_index] = True

            # ====== ログ記録 ======
            newly_unmasked = transfer_index[0].nonzero(as_tuple=True)[0]
            for pos in newly_unmasked:
                pos_int = pos.item()
                if pos_int >= prompt_len:  # 生成領域のみ記録
                    unmask_log.append({
                        "step": global_step,
                        "position": pos_int,
                        "gen_position": pos_int - prompt_len,
                        "token_id": x0[0, pos_int].item(),
                        "confidence": x0_p[0, pos_int].item(),
                    })

            # アンマスク実行
            x[transfer_index] = x0[transfer_index]

    return x, unmask_log, prompt_len

## 5. トークン → 品詞 アライメント

In [ ]:
def extract_content_tokens(x, prompt_len, tokenizer, mask_id):
    """
    生成領域から有効なトークン（EOS/PAD/MASK を除く）を抽出する

    Returns:
        list[dict]: [{position, token_id, token_str}, ...]
    """
    gen_ids = x[0, prompt_len:].tolist()
    eos_id = tokenizer.eos_token_id
    pad_id = tokenizer.pad_token_id

    # EOS/EOT 系の特殊トークンID（LLaDA 固有）
    special_ids = {mask_id}
    if eos_id is not None:
        special_ids.add(eos_id)
    if pad_id is not None:
        special_ids.add(pad_id)
    # LLaDA の EOT token (126081) も除外
    special_ids.add(126081)

    tokens = []
    for i, tid in enumerate(gen_ids):
        if tid in special_ids:
            break  # 最初の特殊トークンで打ち切り
        token_str = tokenizer.decode([tid])
        tokens.append({
            "position": prompt_len + i,
            "gen_position": i,
            "token_id": tid,
            "token_str": token_str,
        })
    return tokens


def align_tokens_with_pos(content_tokens, tokenizer):
    """
    LLaDA の生成トークン列に fugashi の品詞情報を対応付ける

    文字位置の累積でアライメントを取る。

    Returns:
        list[dict]: 各トークンに品詞情報を付加したリスト
            [{position, gen_position, token_id, token_str, pos, word_type}, ...]
    """
    if not content_tokens:
        return []

    # 全テキストを連結
    full_text = "".join(t["token_str"] for t in content_tokens)

    # fugashi 形態素解析
    tagger = fugashi.Tagger()
    morphemes = tagger(full_text)

    # 文字位置ベースのアライメント
    results = []
    morph_idx = 0
    morph_char_pos = 0
    token_char_pos = 0

    for t in content_tokens:
        token_str = t["token_str"]
        token_char_end = token_char_pos + len(token_str)

        # このトークンの範囲にある最初の形態素の品詞を採用
        matched_pos = None
        while morph_idx < len(morphemes) and morph_char_pos < token_char_end:
            m = morphemes[morph_idx]
            if matched_pos is None:
                matched_pos = m.feature.pos1
            morph_char_pos += len(m.surface)
            morph_idx += 1

        pos = matched_pos or "不明"
        word_type = "内容語" if pos in CONTENT_POS else "機能語"

        results.append({
            **t,
            "pos": pos,
            "word_type": word_type,
        })
        token_char_pos = token_char_end

    return results


def merge_unmask_log_with_pos(unmask_log, aligned_tokens):
    """
    アンマスクログとPOS情報を結合する

    Returns:
        list[dict]: 有効トークンのみの結合結果
            [{step, position, gen_position, token_str, confidence, pos, word_type, norm_step}, ...]
    """
    # position → unmask info のマップ
    log_map = {entry["position"]: entry for entry in unmask_log}

    # 最大ステップ（正規化用）
    max_step = max(e["step"] for e in unmask_log) if unmask_log else 1

    merged = []
    for t in aligned_tokens:
        log_entry = log_map.get(t["position"])
        if log_entry:
            merged.append({
                "step": log_entry["step"],
                "norm_step": log_entry["step"] / max_step,  # 0〜1 に正規化
                "position": t["position"],
                "gen_position": t["gen_position"],
                "token_str": t["token_str"],
                "confidence": log_entry["confidence"],
                "pos": t["pos"],
                "word_type": t["word_type"],
            })
    return merged

## 6. 実験実行

In [ ]:
def run_single_experiment(prompt_text, model, tokenizer):
    """1つのプロンプトに対して生成・分析を行う"""
    # プロンプトを chat テンプレートでフォーマット
    messages = [
        {"role": "user", "content": prompt_text},
    ]
    # apply_chat_template がある場合はそれを使用、なければ直接エンコード
    if hasattr(tokenizer, "apply_chat_template"):
        input_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=True
        )
    else:
        text = f"<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n"
        input_ids = tokenizer(text, return_tensors="pt")["input_ids"]

    prompt = input_ids.to(model.device)

    # 生成（ログ付き）
    x, unmask_log, prompt_len = generate_with_logging(
        model, prompt,
        steps=STEPS,
        gen_length=GEN_LENGTH,
        block_length=BLOCK_LENGTH,
        temperature=TEMPERATURE,
        cfg_scale=CFG_SCALE,
        remasking="low_confidence",
        mask_id=MASK_ID,
    )

    # 有効トークンの抽出
    content_tokens = extract_content_tokens(x, prompt_len, tokenizer, MASK_ID)

    # POS アライメント
    aligned = align_tokens_with_pos(content_tokens, tokenizer)

    # ログと POS を結合
    merged = merge_unmask_log_with_pos(unmask_log, aligned)

    # 生成テキスト
    gen_text = tokenizer.decode(
        x[0, prompt_len:], skip_special_tokens=True
    ).strip()

    return {
        "prompt": prompt_text,
        "generated_text": gen_text,
        "token_count": len(content_tokens),
        "merged_data": merged,
    }

In [ ]:
# 全プロンプトで実験実行
all_results = []

for i, prompt_text in enumerate(TEST_PROMPTS):
    print(f"\n{'━' * 60}")
    print(f"[{i+1}/{len(TEST_PROMPTS)}] {prompt_text}")
    print(f"{'━' * 60}")

    result = run_single_experiment(prompt_text, model, tokenizer)
    all_results.append(result)

    print(f"生成文: {result['generated_text']}")
    print(f"有効トークン数: {result['token_count']}")

    # アンマスク順序を表示
    if result["merged_data"]:
        sorted_data = sorted(result["merged_data"], key=lambda d: d["step"])
        print(f"\n  {'Step':>4} {'トークン':<8} {'品詞':<6} {'分類':<6} {'確信度':>8}")
        print(f"  {'-' * 40}")
        for d in sorted_data[:15]:  # 最初の15トークンを表示
            print(f"  {d['step']:4d} {d['token_str']:<8} {d['pos']:<6} "
                  f"{d['word_type']:<6} {d['confidence']:8.4f}")
        if len(sorted_data) > 15:
            print(f"  ... ({len(sorted_data) - 15} tokens omitted)")

print(f"\n{'━' * 60}")
print(f"全 {len(all_results)} プロンプトの実験完了")

## 7. 品詞別集計

In [ ]:
# 全結果をフラットに結合
all_merged = []
for r in all_results:
    all_merged.extend(r["merged_data"])

print(f"分析対象トークン総数: {len(all_merged)}")

# 品詞別の正規化ステップ（0=最初にアンマスク, 1=最後にアンマスク）
pos_stats = defaultdict(lambda: {"norm_steps": [], "confidences": [], "count": 0})

for d in all_merged:
    s = pos_stats[d["pos"]]
    s["norm_steps"].append(d["norm_step"])
    s["confidences"].append(d["confidence"])
    s["count"] += 1

# 集計テーブル
print(f"\n{'品詞':<10} {'平均正規化ステップ':>16} {'平均確信度':>10} {'分類':<6} {'N':>4}")
print("-" * 52)

llada_pos_summary = {}
for pos, s in sorted(pos_stats.items(), key=lambda x: np.mean(x[1]["norm_steps"])):
    avg_step = np.mean(s["norm_steps"])
    avg_conf = np.mean(s["confidences"])
    word_type = "内容語" if pos in CONTENT_POS else "機能語"
    print(f"{pos:<10} {avg_step:16.3f} {avg_conf:10.4f} {word_type:<6} {s['count']:4d}")
    llada_pos_summary[pos] = avg_step

# 内容語 vs 機能語
content_steps = [d["norm_step"] for d in all_merged if d["word_type"] == "内容語"]
function_steps = [d["norm_step"] for d in all_merged if d["word_type"] == "機能語"]

print(f"\n--- 内容語 vs 機能語 ---")
if content_steps:
    print(f"内容語の平均正規化ステップ: {np.mean(content_steps):.3f} (n={len(content_steps)})")
if function_steps:
    print(f"機能語の平均正規化ステップ: {np.mean(function_steps):.3f} (n={len(function_steps)})")

if content_steps and function_steps:
    if np.mean(content_steps) < np.mean(function_steps):
        print("→ LLaDA は内容語を先にアンマスクする傾向")
    else:
        print("→ LLaDA は機能語を先にアンマスクする傾向")

## 8. 可視化

In [ ]:
# ----- 8a: LLaDA POS unmask order -----

sorted_pos = sorted(llada_pos_summary.items(), key=lambda x: x[1])
pos_labels = [p[0] for p in sorted_pos]
pos_values = [p[1] for p in sorted_pos]
pos_colors = ["#e74c3c" if p in CONTENT_POS else "#3498db" for p in pos_labels]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(pos_labels, pos_values, color=pos_colors)
ax.set_xlabel("Normalized Unmask Step (0=first, 1=last)")
ax.set_title("LLaDA: POS-based Unmask Order")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#e74c3c", label="Content words"),
    Patch(facecolor="#3498db", label="Function words"),
]
ax.legend(handles=legend_elements, loc="lower right")

for bar, val in zip(bars, pos_values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("llada_pos_unmask_order.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: llada_pos_unmask_order.png")


In [ ]:
# ----- 8b: BERT vs LLaDA comparison -----

common_pos = sorted(
    set(BERT_BASELINE.keys()) & set(llada_pos_summary.keys()),
    key=lambda p: BERT_BASELINE.get(p, 1)
)

if common_pos:
    bert_vals = [BERT_BASELINE[p] for p in common_pos]
    llada_vals = [llada_pos_summary[p] for p in common_pos]

    x_pos = np.arange(len(common_pos))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x_pos - width/2, bert_vals, width, label="BERT (Exp1)",
                   color="#2ecc71", alpha=0.85)
    bars2 = ax.bar(x_pos + width/2, llada_vals, width, label="LLaDA (Exp2)",
                   color="#9b59b6", alpha=0.85)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(common_pos)
    ax.set_ylabel("Normalized Unmask Step (0=first, 1=last)")
    ax.set_title("BERT vs LLaDA: POS-based Unmask Order Comparison")
    ax.legend()

    for i, pos in enumerate(common_pos):
        if pos in CONTENT_POS:
            ax.axvspan(i - 0.45, i + 0.45, alpha=0.08, color="red")
        else:
            ax.axvspan(i - 0.45, i + 0.45, alpha=0.08, color="blue")

    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f"{h:.2f}", ha="center", va="bottom", fontsize=8)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f"{h:.2f}", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    plt.savefig("bert_vs_llada_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: bert_vs_llada_comparison.png")
else:
    print("No common POS tags found. Check LLaDA generation results.")


In [ ]:
# ----- 8c: Per-sentence unmask heatmap -----

for i, result in enumerate(all_results):
    merged = result["merged_data"]
    if not merged:
        continue

    sorted_data = sorted(merged, key=lambda d: d["gen_position"])
    tokens = [d["token_str"] for d in sorted_data]
    steps = [d["norm_step"] for d in sorted_data]
    pos_list = [d["pos"] for d in sorted_data]

    fig, ax = plt.subplots(figsize=(max(12, len(tokens) * 0.5), 2))
    colors = plt.cm.RdYlGn_r(steps)

    for j, (tok, step, pos) in enumerate(zip(tokens, steps, pos_list)):
        ax.barh(0, 1, left=j, color=colors[j], edgecolor="white", linewidth=0.5)
        # token text (may be JP, ok if garbled in heatmap - color is the info)
        ax.text(j + 0.5, 0, tok, ha="center", va="center", fontsize=7,
                fontweight="bold")
        ax.text(j + 0.5, -0.6, pos, ha="center", va="center", fontsize=6,
                color="gray")

    ax.set_xlim(0, len(tokens))
    ax.set_ylim(-1.2, 0.8)
    ax.axis("off")
    ax.set_title(f"Unmask order (green=early, red=late) | prompt {i+1}", fontsize=9)

    plt.tight_layout()
    plt.savefig(f"llada_heatmap_{i+1}.png", dpi=150, bbox_inches="tight")
    plt.show()


## 9. 結果を CSV で保存

In [ ]:
# ----- 9a: 全トークンの詳細データ -----
rows = []
for result in all_results:
    for d in result["merged_data"]:
        rows.append({
            "prompt": result["prompt"],
            "generated_text": result["generated_text"],
            "step": d["step"],
            "norm_step": d["norm_step"],
            "gen_position": d["gen_position"],
            "token": d["token_str"],
            "confidence": d["confidence"],
            "pos": d["pos"],
            "word_type": d["word_type"],
        })

df_detail = pd.DataFrame(rows)
df_detail.to_csv("llada_unmask_detail.csv", index=False, encoding="utf-8-sig")
print(f"保存: llada_unmask_detail.csv ({len(df_detail)} rows)")
display(df_detail.head(10))

# ----- 9b: 品詞別サマリー -----
summary_rows = []
for pos, s in pos_stats.items():
    word_type = "内容語" if pos in CONTENT_POS else "機能語"
    summary_rows.append({
        "model": "LLaDA",
        "pos": pos,
        "word_type": word_type,
        "mean_norm_step": np.mean(s["norm_steps"]),
        "mean_confidence": np.mean(s["confidences"]),
        "count": s["count"],
    })

# BERT のデータも追加（比較用）
for pos, norm_step in BERT_BASELINE.items():
    word_type = "内容語" if pos in CONTENT_POS else "機能語"
    summary_rows.append({
        "model": "BERT",
        "pos": pos,
        "word_type": word_type,
        "mean_norm_step": norm_step,
        "mean_confidence": None,
        "count": None,
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv("llada_bert_pos_summary.csv", index=False, encoding="utf-8-sig")
print(f"\n保存: llada_bert_pos_summary.csv ({len(df_summary)} rows)")
display(df_summary)

## 10. 結果まとめ

### Research Question

拡散型言語モデル（LLaDA）は、確信度順アンマスクにおいて **内容語（名詞・動詞）と機能語（助詞・助動詞）のどちらを先に復元するか？**

### 比較

| モデル | タイプ | アンマスク傾向 |
| ------ | ------ | -------------- |
| BERT (Exp1) | マスク言語モデル (430MB) | 助詞（機能語）を先に復元 |
| LLaDA (Exp2) | マスク拡散型 LLM (8B) | ← 実験結果を記入 |

### 考察ポイント

- BERT と LLaDA で傾向は一致するか？ → 構文優先は普遍的な性質か
- モデルサイズの影響: 430MB vs 8B で順序は変わるか
- 拡散型の特性: LLaDA は BERT と異なるアンマスク戦略を学習しているか
- SeedLM への示唆: 拡散型でも「意味の核→構文」の逆順生成は必要か